In [1]:
#!pip install python-dotenv snowflake-connector-python

import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import time

load_dotenv()

# Strava credentials
client_id = os.getenv('STRAVA_CLIENT_ID')
client_secret = os.getenv('STRAVA_CLIENT_SECRET')

# Snowflake credentials
snowflake_account = os.getenv('SNOWFLAKE_ACCOUNT')
snowflake_user = os.getenv('SNOWFLAKE_USER')
snowflake_password = os.getenv('SNOWFLAKE_PASSWORD')
snowflake_warehouse = os.getenv('SNOWFLAKE_WAREHOUSE')
snowflake_database = os.getenv('SNOWFLAKE_DATABASE')
snowflake_schema = os.getenv('SNOWFLAKE_SCHEMA')

In [2]:
# Generate authorization URL
auth_url = (
    f"https://www.strava.com/oauth/authorize"
    f"?client_id={client_id}"
    f"&response_type=code"
    f"&redirect_uri=http://localhost/exchange_token"
    f"&approval_prompt=force"
    f"&scope=activity:read_all"
)

print(auth_url)

https://www.strava.com/oauth/authorize?client_id=83239&response_type=code&redirect_uri=http://localhost/exchange_token&approval_prompt=force&scope=activity:read_all


In [3]:
# Exchange authorization code for access token
code = "61e2d72523e36ed69d922d25d6e00c94590068ae"

token_response = requests.post(
    "https://www.strava.com/oauth/token",
    data={
        "client_id": client_id,
        "client_secret": client_secret,
        "code": code,
        "grant_type": "authorization_code"
    }
)

token_data = token_response.json()
access_token = token_data["access_token"]

In [4]:
# Pull all activities from Strava API
url = "https://www.strava.com/api/v3/athlete/activities"
headers = {"Authorization": f"Bearer {access_token}"}

all_activities = []
page = 1

while True:
    params = {
        'per_page': 200,
        'page': page
    }
    response = requests.get(url, headers=headers, params=params)
    activities = response.json()
    
    all_activities = all_activities + activities
    
    if len(activities) < 200:
        break
    
    page = page + 1

strava_df = pd.DataFrame(all_activities)
strava_df.shape

(4945, 58)

In [5]:
strava_df.to_csv('strava_raw.csv', index=False)

In [6]:
pd.set_option('display.max_columns', None)
strava_df.head()

,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,device_name,id,start_date,start_date_local,timezone,utc_offset,location_city,location_state,location_country,achievement_count,kudos_count,comment_count,athlete_count,photo_count,map,trainer,commute,manual,private,visibility,flagged,gear_id,start_latlng,end_latlng,average_speed,max_speed,has_heartrate,heartrate_opt_out,display_hide_heartrate_option,elev_high,elev_low,upload_id,upload_id_str,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,average_cadence,average_watts,max_watts,weighted_average_watts,device_watts,kilojoules,average_heartrate,max_heartrate,suffer_score,average_temp
0,2,"{'id': 30968453, 'resource_state': 1}",Morning Workout,0.0,598,598,0.0,Workout,Workout,NaN,COROS PACE 3,17601398967,2026-03-04T14:42:13Z,2026-03-04T08:42:13Z,(GMT-06:00) America/Bahia_Banderas,-21600.0,None,None,None,0,0,0,1,0,"{'id': 'a17601398967', 'summary_polyline': '',...",True,False,False,True,only_me,False,NaN,[],[],0.000,0.0,False,False,False,0.0,0.0,1.869931e+10,18699310375,475838336945127624.fit,False,0,0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"{'id': 30968453, 'resource_state': 1}",🎶 Put the paper in a picture like a diorama 🎶,10089.8,3111,3336,33.0,Run,Run,0.0,COROS PACE 3,17601170525,2026-03-04T13:32:54Z,2026-03-04T07:32:54Z,(GMT-06:00) America/Chicago,-21600.0,None,None,None,0,4,0,1,0,"{'id': 'a17601170525', 'summary_polyline': 'cr...",False,False,False,False,everyone,False,g24756529,"[41.884669, -87.626326]","[41.885201, -87.626152]",3.243,4.0,True,False,True,199.0,180.0,1.869908e+10,18699079862,475837964612567142.fit,False,0,0,False,84.7,272.1,676.0,265.0,True,846.4,143.5,160.0,16.0,NaN
2,2,"{'id': 30968453, 'resource_state': 1}",Morning Workout,0.0,419,419,0.0,Workout,Workout,NaN,COROS PACE 3,17600560739,2026-03-04T13:20:48Z,2026-03-04T07:20:48Z,(GMT-06:00) America/Bahia_Banderas,-21600.0,None,None,None,0,0,0,1,0,"{'id': 'a17600560739', 'summary_polyline': '',...",True,False,False,True,only_me,False,NaN,[],[],0.000,0.0,True,False,True,0.0,0.0,1.869849e+10,18698489656,475837043344179400.fit,False,0,0,False,NaN,NaN,NaN,NaN,NaN,NaN,84.5,105.0,0.0,NaN
3,2,"{'id': 30968453, 'resource_state': 1}",Ladder: Let Those Hips Breathe,0.0,1029,1029,0.0,Workout,Workout,NaN,Ladder,17595950591,2026-03-04T02:17:03Z,2026-03-04T04:17:03Z,(GMT+02:00) Africa/Harare,7200.0,None,None,None,0,0,0,1,0,"{'id': 'a17595950591', 'summary_polyline': '',...",True,False,False,True,only_me,False,NaN,[],[],0.000,0.0,False,False,False,0.0,0.0,1.869385e+10,18693852787,ee9d94ca-2513-4803-a198-5f1d5f1e5d0e.fit,False,0,0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,"{'id': 30968453, 'resource_state': 1}",Morning Workout,0.0,839,839,0.0,Workout,Workout,NaN,COROS PACE 3,17588931510,2026-03-03T15:00:17Z,2026-03-03T09:00:17Z,(GMT-06:00) America/Bahia_Banderas,-21600.0,None,None,None,0,0,0,1,0,"{'id': 'a17588931510', 'summary_polyline': '',...",True,False,False,True,only_me,False,NaN,[],[],0.000,0.0,True,False,True,0.0,0.0,1.868683e+10,18686825694,475815500322996227.fit,False,0,0,False,NaN,NaN,NaN,NaN,NaN,NaN,78.5,103.0,1.0,NaN


In [7]:
strava_df.columns.tolist()

['resource_state',
 'athlete',
 'name',
 'distance',
 'moving_time',
 'elapsed_time',
 'total_elevation_gain',
 'type',
 'sport_type',
 'workout_type',
 'device_name',
 'id',
 'start_date',
 'start_date_local',
 'timezone',
 'utc_offset',
 'location_city',
 'location_state',
 'location_country',
 'achievement_count',
 'kudos_count',
 'comment_count',
 'athlete_count',
 'photo_count',
 'map',
 'trainer',
 'commute',
 'manual',
 'private',
 'visibility',
 'flagged',
 'gear_id',
 'start_latlng',
 'end_latlng',
 'average_speed',
 'max_speed',
 'has_heartrate',
 'heartrate_opt_out',
 'display_hide_heartrate_option',
 'elev_high',
 'elev_low',
 'upload_id',
 'upload_id_str',
 'external_id',
 'from_accepted_tag',
 'pr_count',
 'total_photo_count',
 'has_kudoed',
 'average_cadence',
 'average_watts',
 'max_watts',
 'weighted_average_watts',
 'device_watts',
 'kilojoules',
 'average_heartrate',
 'max_heartrate',
 'suffer_score',
 'average_temp']

In [8]:
strava_df['average_temp'].notna().sum()

np.int64(2011)

In [9]:
strava_df['start_latlng'].value_counts().head(10)

start_latlng
[]                          2413
[41.884669, -87.626326]        1
[41.884078, -87.62548]         1
[-21.714224, 166.155639]       1
[41.884796, -87.626555]        1
[41.884474, -87.625644]        1
[41.884059, -87.624549]        1
[33.825273, -116.550552]       1
[41.884624, -87.625995]        1
[41.792608, -87.596362]        1
Name: count, dtype: int64

In [10]:
# Connect to Snowflake
conn = snowflake.connector.connect(
    account=snowflake_account,
    user=snowflake_user,
    password=snowflake_password,
    warehouse=snowflake_warehouse,
    database=snowflake_database,
    schema=snowflake_schema
)

cursor = conn.cursor()

In [11]:
# Drop existing table if it exists
cursor.execute("DROP TABLE IF EXISTS STRAVA_ACTIVITIES")

# Prepare dataframe for Snowflake
strava_upload = strava_df.copy()
strava_upload.columns = strava_upload.columns.str.upper()

# Write to Snowflake
write_pandas(conn, strava_upload, 'STRAVA_ACTIVITIES', auto_create_table=True)

(True,
 1,
 4945,
 [('snowpark_temp_stage_4bfx44jgja/file0.txt',
   'LOADED',
   4945,
   4945,
   1,
   0,
   None,
   None,
   None,
   None)])

In [12]:
# Need coordinates split to pull weather data
weather_df = strava_df[
    (strava_df['start_latlng'].apply(lambda x: len(x) == 2)) &
    (strava_df['type'] != 'VirtualRide')
].copy()

# Extract lat and lng
weather_df['lat'] = weather_df['start_latlng'].apply(lambda x: x[0])
weather_df['lng'] = weather_df['start_latlng'].apply(lambda x: x[1])

In [13]:
weather_df[['start_latlng', 'lat', 'lng']].head()

,start_latlng,lat,lng
1,"[41.884669, -87.626326]",41.884669,-87.626326
5,"[41.884078, -87.62548]",41.884078,-87.625480
12,"[41.884796, -87.626555]",41.884796,-87.626555
14,"[41.884474, -87.625644]",41.884474,-87.625644
16,"[41.884059, -87.624549]",41.884059,-87.624549


In [14]:
def get_weather(lat, lng, date, hour):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lng,
        "start_date": date,
        "end_date": date,
        "hourly": ["temperature_2m", "apparent_temperature",
                   "precipitation", "windspeed_10m",
                   "relativehumidity_2m"],
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph",
        "precipitation_unit": "inch",
        "timezone": "auto"
    }
    
    for attempt in range(3):  # Try up to 3 times
        try:
            response = requests.get(url, params=params, timeout=10)
            data = response.json()
            
            if "hourly" in data:
                hourly = data["hourly"]
                return {
                    "temperature": hourly["temperature_2m"][hour],
                    "feels_like": hourly["apparent_temperature"][hour],
                    "precipitation": hourly["precipitation"][hour],
                    "wind_speed": hourly["windspeed_10m"][hour],
                    "humidity": hourly["relativehumidity_2m"][hour]
                }
        except:
            time.sleep(2)  # Wait 2 seconds before retrying
    
    return None

In [15]:
# Extract date and hour for Open-Meteo hourly lookup
weather_df['start_date_local'] = pd.to_datetime(weather_df['start_date_local'])
weather_df['activity_date'] = weather_df['start_date_local'].dt.date.astype(str)
weather_df['activity_hour'] = weather_df['start_date_local'].dt.hour

weather_df[['start_date_local', 'activity_date', 'activity_hour']].head()

,start_date_local,activity_date,activity_hour
1,2026-03-04 07:32:54+00:00,2026-03-04,7
5,2026-03-03 07:27:15+00:00,2026-03-03,7
12,2026-02-28 08:49:43+00:00,2026-02-28,8
14,2026-02-25 08:28:12+00:00,2026-02-25,8
16,2026-02-24 18:13:29+00:00,2026-02-24,18


In [16]:
# Test on 5 rows first
weather_results_test = []

for _, row in weather_df.head(5).iterrows():
    result = get_weather(row['lat'], row['lng'], row['activity_date'], row['activity_hour'])
    
    if result:
        result['id'] = row['id']
        result['activity_date'] = row['activity_date']
        weather_results_test.append(result)
    
    time.sleep(0.1)

weather_test_df = pd.DataFrame(weather_results_test)
weather_test_df

,temperature,feels_like,precipitation,wind_speed,humidity,id,activity_date
0,33.5,27.3,0.0,5.5,91,17601170525,2026-03-04
1,32.6,26.6,0.0,3.9,84,17588720832,2026-03-03
2,35.0,27.5,0.0,6.2,64,17554937041,2026-02-28
3,28.9,18.9,0.0,10.6,61,17518869065,2026-02-25
4,35.9,25.5,0.0,12.2,54,17512749314,2026-02-24


In [17]:
weather_results = []
checkpoint_file = 'weather_checkpoint.csv'

# Load existing checkpoint if it exists
if os.path.exists(checkpoint_file):
    checkpoint_df = pd.read_csv(checkpoint_file)
    completed_ids = set(checkpoint_df['id'].astype(str))
    weather_results = checkpoint_df.to_dict('records')
    print(f"Resuming from checkpoint: {len(completed_ids)} activities already done")
else:
    completed_ids = set()
    print("Starting fresh")

# Loop through activities
for _, row in weather_df.iterrows():
    
    # Skip if already completed
    if str(row['id']) in completed_ids:
        continue
    
    result = get_weather(row['lat'], row['lng'], row['activity_date'], row['activity_hour'])
    
    if result:
        result['id'] = row['id']
        result['activity_date'] = row['activity_date']
        weather_results.append(result)
        completed_ids.add(str(row['id']))
    
    # Save checkpoint every 100 activities
    if len(weather_results) % 100 == 0:
        pd.DataFrame(weather_results).to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved: {len(weather_results)} activities completed")
    
    time.sleep(0.1)

# Final save
weather_data_df = pd.DataFrame(weather_results)
pd.DataFrame(weather_results).to_csv(checkpoint_file, index=False)
weather_data_df.shape

Starting fresh
Checkpoint saved: 100 activities completed
Checkpoint saved: 200 activities completed
Checkpoint saved: 300 activities completed
Checkpoint saved: 400 activities completed
Checkpoint saved: 500 activities completed
Checkpoint saved: 600 activities completed
Checkpoint saved: 700 activities completed
Checkpoint saved: 800 activities completed
Checkpoint saved: 900 activities completed
Checkpoint saved: 1000 activities completed
Checkpoint saved: 1100 activities completed
Checkpoint saved: 1200 activities completed
Checkpoint saved: 1300 activities completed
Checkpoint saved: 1400 activities completed
Checkpoint saved: 1500 activities completed
Checkpoint saved: 1600 activities completed
Checkpoint saved: 1700 activities completed
Checkpoint saved: 1800 activities completed
Checkpoint saved: 1900 activities completed
Checkpoint saved: 2000 activities completed
Checkpoint saved: 2100 activities completed
Checkpoint saved: 2200 activities completed
Checkpoint saved: 2300 act

(2450, 7)

In [18]:
# Upload weather data to Snowflake
weather_upload = weather_data_df.copy()
weather_upload.columns = weather_upload.columns.str.upper()

write_pandas(conn, weather_upload, 'WEATHER_DATA', auto_create_table=True)

(True,
 1,
 2450,
 [('snowpark_temp_stage_a4f06tric1/file0.txt',
   'LOADED',
   2450,
   2450,
   1,
   0,
   None,
   None,
   None,
   None)])

In [19]:
conn.close()